# Ray Data Patterns and Anti-Patterns
© 2025, Anyscale. All Rights Reserved


This notebook will provide an overview of common Ray Data patterns and anti-patterns to help you build efficient data pipelines.

<div class="alert alert-block alert-info">
<b>Here is the roadmap for this notebook:</b>
<ol>
    <li>Quality of life improvements</li>
    <li>Reading/ingesting data </li>
    <li>Resource management</li>
    <li>Types of Data Operations</li>
    <li>One-to-one data operations</li>
    <li>All-to-all data operations</li>
    <li>Materializing intermediate results</li>
    <li>Writing data</li>
</ol>
</div>

**Imports** 

In [1]:
import os
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import ray
import time

**Constants**

In [2]:
MiB = 1024 ** 2
GiB = 1024 ** 3

# Path to read data
READ_DATA_PATH = "s3://anyscale-public-materials"
DATA_PATH_FEATURES = f"{READ_DATA_PATH}/nyc-taxi-cab-features/"
DATA_PATH_TARGET = f"{READ_DATA_PATH}/nyc-taxi-cab-target/"

# Path to write data (feel free to change to any remote storage path - no need to use Anyscale)
WRITE_DATA_PATH = os.environ["ANYSCALE_ARTIFACT_STORAGE"]

# Use hash-based shuffle strategy (later we explain why and what this does)
ctx = ray.data.DataContext.get_current()
ctx.shuffle_strategy = ray.data.context.ShuffleStrategy.HASH_SHUFFLE

## 1. Quality of life improvements

### 1.1 Name Datasets

It is a good practice to set the name of the dataset in order to easily identify it in metrics and logs.

In [ ]:
ds = ray.data.read_parquet(DATA_PATH_FEATURES)
ds = ds.set_name("take-small-sample")
ds.take()

### 1.2 Configure progress bars appropriately

Depending on your use case, you may not be interested in the full progress bar output, or wish to turn them off altogether. Ray Data provides several ways to accomplish this:

- Disabling operator-level progress bars: Set `DataContext.get_current().enable_operator_progress_bars = False`. This only shows the global progress bar, and omits operator-level progress bars.

- Disabling all progress bars: Set `DataContext.get_current().enable_progress_bars = False`. This disables all progress bars from Ray Data related to dataset execution.

- Disabling ray_tqdm: Set `DataContext.get_current().use_ray_tqdm = False`. This configures Ray Data to use the base tqdm library instead of the custom distributed tqdm implementation, which could be useful when debugging logging issues in a distributed setting.


## 2. Reading Data

Here are some tips for reading data with Ray Data


### 2.1 Handling transient IO errors

Let's go over some common transient IO errors when scaling up reads

If Ray is submitting too many tasks to the cluster, it may be rate-limited by the data source, or encounter other IO errors (e.g. connection erros or timeouts).

Symptoms:
- Transient errors that go away after retrying, backing off or running the same code against a smaller dataset.

Solutions:
- Ensure the errors are being retried by Ray Data.
- Increase the number of retries in the Ray Data read function.

By default, Ray Data will retry on these IO errors.


In [ ]:
ctx = ray.data.DataContext.get_current()
ctx.retried_io_errors

You can append any custom error messages you'd like to retry on as well.

```python
ctx.retried_io_errors.append("AWS Error Access Denied")
```


Note that by default, Ray Data will retry up to 10 max attempts and will implement a backoff strategy.

#### Handling corrupted files

If the files are corrupted, you may see errors attempting to read files.

If you expect to encounter corrupted files, you can use the `max_errored_blocks` parameter to ignore a certain set size of data. (Note a block by default is around 1-128MB)


### 2.2 Reading from parquet

Parquet is a common data source for ML workloads and is very popularly used with Ray Data.

Let's explore common patterns/anti-patterns when working with parquet

#### 1.2.1 Leverage Column Pruning when possible

As a best practice, leverage column pruning to reduce the amount of data read.

i.e. **don't do this** if you only care for a subset of the columns 

In [ ]:
ds_features = ray.data.read_parquet(DATA_PATH_FEATURES)

instead, **do this**

In [5]:
features = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "tolls_amount",
    "total_amount",
    "id",
    "year",
    "month",
]

ds_features = ray.data.read_parquet(DATA_PATH_FEATURES, columns=features)

You can additionally specify a "schema on read" to perform both:
1. column-level pruning
2. ensure that the data types are as expected.

Here is how to do so

In [6]:
schema = pa.schema(
    [
        pa.field("VendorID", pa.int64()),
        pa.field("tpep_pickup_datetime", pa.timestamp("ns")),
        pa.field("tpep_dropoff_datetime", pa.timestamp("ns")),
        pa.field("passenger_count", pa.int64()),
        pa.field("trip_distance", pa.float64()),
        pa.field("RatecodeID", pa.int64()),
        pa.field("store_and_fwd_flag", pa.string()),
        pa.field("PULocationID", pa.int64()),
        pa.field("DOLocationID", pa.int64()),
        pa.field("payment_type", pa.int64()),
        pa.field("fare_amount", pa.float64()),
        pa.field("extra", pa.float64()),
        pa.field("mta_tax", pa.float64()),
        pa.field("tolls_amount", pa.float64()),
        pa.field("improvement_surcharge", pa.float64()),
        pa.field("total_amount", pa.float64()),
        pa.field("congestion_surcharge", pa.float64()),
        pa.field("airport_fee", pa.float64()),
        pa.field("id", pa.int64()),
        pa.field("year", pa.string()),
        pa.field("month", pa.string()),
    ]
)

df_features = ray.data.read_parquet(DATA_PATH_FEATURES, schema=schema)

#### 1.2.2 Perform efficient data selection and filtering

#### Filtering on hive partitioned columns

Hive partitioning is a partitioning strategy that is used to split a table into multiple files based on partition keys.  As example, our dataset is organized as follows:

```
DATA_PATH_FEATURES/
├── year=2011/
│   ├── month=1/
│   ├── month=2/
│   └── ...
├── year=2012/
│   ├── month=1/
│   ├── month=2/
│   └── ...
├── year=2013/
│   ├── month=1/
│   ├── month=2/
│   └── ...
```

If the parquet directory is organized under a partitioned directory structure, you can use the `partition_filter` parameter to filter out the partitions you don't want to read.

Here is how to implement a calendar-based split on the training and validation datasets.

In [7]:
from datetime import datetime
from functools import partial
from typing import Any

from ray.data.datasource.partitioning import (
    PathPartitionFilter,
    PathPartitionParser,
    Partitioning,
    PartitionStyle,
)

def filter_fn(values: dict[str, Any], start_date: datetime, end_date: datetime) -> bool:
    dt = datetime.strptime(values["year"] + "-" + values["month"], "%Y-%m")
    return start_date <= dt <= end_date

filter_train = PathPartitionFilter(
    filter_fn=partial(filter_fn, start_date=datetime(2017, 10, 1), end_date=datetime(2017, 12, 31)),
    path_partition_parser=PathPartitionParser(
        partitioning=Partitioning(
            style=PartitionStyle.HIVE,
            field_names=["year", "month"],
            field_types={
                "year": pa.int64(),
                "month": pa.int64(),
            },
        )
    ),
)

In [8]:
ds_features_train = ray.data.read_parquet(
    DATA_PATH_FEATURES,
    partition_filter=filter_train
)

Let's count the number of rows in the training dataset:

Note, how efficient this is

In [ ]:
ds_features_train.set_name("train_features_count")
ds_features_train.count()

#### Activity on data filtering

<div class="alert alert-info"

Generate a validation dataset keeping rows for the year 2018 only.

```python
df_features_valid = ... # TODO

# then count the number of rows
ds_features_valid.set_name("valid_features_count")
num_rows_valid = ds_features_valid.count()
print(f"Number of validing features: {num_rows_valid:,}")
```

</div>

In [10]:
# write your solution here


<div class="alert alert-block alert-info">

<details>

<summary> Click here to see the solution </summary>

```python
filter_valid = PathPartitionFilter(
    filter_fn=partial(filter_fn, start_date=datetime(2018, 1, 1), end_date=datetime(2018, 12, 31)),
    path_partition_parser=PathPartitionParser(
        partitioning=Partitioning(
            style=PartitionStyle.HIVE,
            field_names=["year", "month"],
            field_types={
                "year": pa.int64(),
                "month": pa.int64(),
            },
        )
    ),
)

ds_features_valid = ray.data.read_parquet(
    DATA_PATH_FEATURES,
    columns=features,
    partition_filter=filter_valid
)


# then count the number of rows
ds_features_valid.set_name("valid_features_count")
num_rows_valid = ds_features_valid.count()
print(f"Number of validing features: {num_rows_valid:,}")
```

</details>

Let's start by performing a calendar split to split the data into training, validation, and test sets.

#### Parquet File Format

Here is a diagram of a parquet file layout taken from the [parquet docs](https://parquet.apache.org/docs/file-format/):

<img src="https://parquet.apache.org/images/FileLayout.gif" alt="Parquet File Layout" width="500" loading="lazy">

#### Filtering parquet

##### Filtering on non-partitioned columns

To filter on a non-partitioned columns, the parquet reader uses `filter` to determine which row groups or pages to read. Under the hood, the filter is pushed down to the parquet reader - i.e. not applied after all the data is read.

##### Filter Pushdown in Parquet
**Row Group Pruning**
Parquet files [store metadata](https://parquet.apache.org/docs/file-format/metadata/) (statistics, encoding, type, compression) for each column in every row group, and readers evaluate filter predicates against these stats to skip entire row groups that cannot contain matching rows, greatly reducing I/O 

**Page-Level Pruning**
With page indexes enabled, readers use per-page statistics to skip individual data pages within row groups when their ranges don’t satisfy the filter, offering finer-grained I/O savings 

<div class="alert alert-info">

When reading parquet files and filtering on a non-partitioned column, do one of the following:
- Specify a filter expression as part of `read_parquet`
- Use `.filter` with an expression immediately after the read to perform a predicate pushdown.

</div>

Here is an example of a filter expression on `payment_type`

In [ ]:
def get_filter_expression(payment_types: list[int]) -> pc.Expression:
    return pa.dataset.field("payment_type").isin(payment_types)

get_filter_expression(
    payment_types=[
        1,  # Credit card
        2,  # Cash
    ]
)

Here is how it can be applied as part of the read to **ensure pushdown**:

In [12]:
ds_valid_payment_types = ray.data.read_parquet(DATA_PATH_FEATURES, columns=features, filter=get_filter_expression([1, 2]), partition_filter=filter_train)

Counting the rows now will only show `Read` operations involved (i.e no separate `Filter` stage/operation):

In [ ]:
ds_valid_payment_types.set_name("get_filtered_count")
valid_payment_types_count = ds_valid_payment_types.count()
print(f"Number of rows with valid payment types: {valid_payment_types_count:,}")

i.e. **don't do this**

```python
def filter_func(row):
    return row["payment_type"] in [1, 2]

ds_inefficent = ray.data.read_parquet(DATA_PATH_FEATURES, columns=features).filter(fn=filter_func)
ds_inefficent.set_name("inefficient_filter")
cash_payments_count_inefficient = ds_inefficent.count()
print(f"Number of rows with cash payments: {cash_payments_count_inefficient:,}")
```

Because Ray Data will first perform the read, then perform the filter **row by row** reading each block and applying the filter function.

## 3. Resource Management

### Ray Data Resource Management and Limits

Ray Data uses a global resource manager to manage resource limits for dataset execution. 

#### Understand the default behavior

By default, Ray Data aggressively scales up the cluster to submit tasks, more specifically here is how the resource manager computes its available resources:

- **Object Store Capacity**: By default, the resource manager limits usage to a portion of the object store capacity
- **CPU and GPU Resources**: The resource manager will utilize all available CPU and GPU resources on the cluster by default.

In [ ]:
from ray.data._internal.execution.resource_manager import ResourceManager

ResourceManager.DEFAULT_OBJECT_STORE_MEMORY_LIMIT_FRACTION 

#### Customize Resource Limits

There are several approaches to control and limit the resources used by Ray Data.

##### 1. Setting Cluster-wide Resource Limits

Define an upper limit on the compute configuration using the compute/cluster configuration file to avoid unbounded auto-scaling.

As an example, set the max number of nodes to 2 if you want to avoid further scaling up the cluster.

```yaml
compute_config:
    ...
    worker_nodes:
        ...
        max_nodes: 2
```


##### 2. Setting Global Resource Limits 
You can customize resource limits using the `DataContext`:

- **Object Store Memory Limit**:
  
  ```python
  ctx = ray.data.DataContext.get_current()
  ctx.execution_options.resource_limits.object_store_memory = 10e9  # 10 GiB
  ```

- **CPU and GPU Limits**:
  ```python
  ctx = ray.data.DataContext.get_current()
  ctx.execution_options.resource_limits.cpu = 10
  ctx.execution_options.resource_limits.gpu = 5
  ```

- **Excluding Resources**:
  ```python
  ctx = ray.data.DataContext.get_current()
  ctx.execution_options.exclude_resources = ExecutionResources(cpu=10, gpu=5)  # Exclude 10 CPUs and 5 GPUs
  ```

See the [docs for more details](https://docs.ray.io/en/latest/data/execution-configurations.html)


In [ ]:
ctx = ray.data.DataContext.get_current()

# by default there are no resource limits for Ray Data 
ctx.execution_options.resource_limits.cpu, ctx.execution_options.resource_limits.gpu, ctx.execution_options.resource_limits.memory

To verify that limiting compute resources affects dataset execution, run the below script.

In [ ]:
!python resource_control.py

##### 3. Setting Per-Operator Resource Limits
You can set per-operator concurrency constraints, for example:
- Limit concurrency for each operator.
- Adjust the ratio of files to metadata tasks.

More specifically:
- Specify per-operator concurrency with `.map_batches(, concurrency=X)`.
- If running on OSS Ray: For metadata fetching, adjust `ray.data.datasource.parquet_meta_provider.FRAGMENTS_PER_META_FETCH = N` (default is 6; doubling it reduces tasks by half).


#### Avoid disparities in node sizes

Ray Data calculates the object store memory budget based on the cluster's total global memory.

Disparities in worker node memory can cause disk spilling on the worker nodes with smaller memory.

## 3. Types of Data Operations

Let's go over how Ray Data categorizes its data operations 

### Categorizing Operations in Ray Data

In Ray Data, operations are categorized based on data movement patterns and transformations. The three main categories are:

**1. OneToOneOperator**

- **Definition:** Each input record maps directly to one output record, with no data movement across nodes.
- **Characteristics:** Local to each node, highly efficient, no network I/O.
- **Examples:** 
  - **Map:** Transforms each element independently.
  - **Filter:** Retains elements based on a condition.
  - **FlatMap:** Maps an input to zero or more outputs.
  - **MapBatches:** Applies a transformation to each batch independently.

**2. AllToAllOperator**

- **Definition:** Data from all nodes is redistributed, and each output record may depend on input records from any node.
- **Characteristics:** Involves network data shuffling, expensive due to network I/O and serialization/deserialization.
- **Examples:** 
  - **GroupBy:** Groups records by key.
  - **Sort:** Ensures global order by redistributing data.

**3. NAryOperator**

- **Definition:** Combines multiple datasets or input streams into one operation, operating on multiple input datasets simultaneously.
- **Characteristics:** Can involve both OneToOne and AllToAll behaviors, complexity increases with the number of inputs.
- **Examples:** 
  - **Union:** Combines multiple datasets into one.
  - **Zip:** Aligns elements from multiple datasets.

## 4. One-to-one data operations

One-to-one data operations are operations that map each input record to one output record.

These operations are highly parallelizable and can be executed in a distributed manner.

Let's create a simple function to preprocess the data.

In [17]:
def compute_tolls_pct(df: pd.DataFrame) -> pd.DataFrame:
    df["tolls_pct"] = df["tolls_amount"] / df["total_amount"]
    return df

We can take the same function and apply it to the Ray dataset using `map_batches`. 

`map_batches` will batch each block of the dataset and apply the function to each batch in parallel.

In [ ]:
ds = ray.data.read_parquet(DATA_PATH_FEATURES, columns=["tolls_amount", "total_amount"])
ds_adjusted = ds.map_batches(compute_tolls_pct, batch_format="pandas")
ds_adjusted.take()

#### How `map_batches` works under the hood

#### Choosing between tasks and actors
Depending on the nature of the transformation, Ray Data will choose one of the two compute strategies:
- `Task` based execution in case the transformation is **stateless** - i.e. defined as a python function
- `Actor` based execution in case the transformation is **stateful** - i.e. defined as a callable class

#### Determining concurrency

- If a `Task` based execution is chosen, Ray Data's scheduler will dynamically allocate the number of tasks to run based on the dynamic resource budget. If `concurrency` is set, it will be treated as the maximum number of tasks to run.
- If an `Actor` based execution is chosen, Ray Data will use a fixed number of actors equal to the `concurrency` parameter.

#### Batching behavior

A block is processed by:
- Batching it using the `batch_size` parameter
- Sequentially applying the transformation to each batch
- Forming an output block from the results once the target block size is reached
- Yielding the output block to the object store

#### Batch format:

The batch format can be set as either:
- `pandas` - a pandas dataframe
- `pyarrow` - a pyarrow table
- `numpy` - a numpy array

NOTE: only use `pandas` if you need it. `pandas` usually incurs a performance penalty in terms of memory and CPU.

Regardless of the batch format, a block will continue to be serialized as a pyarrow table in most cases.

### Preventing out-of-memory errors

Ray Data's scheduler does not account for heap memory usage.

To prevent excessive task submissions, set memory requirements for each operator. 

For example:

```python
def add_column(batch: dict) -> dict:
    batch["b"] = batch["a"] + 1
    return batch

ds.map_batches(add_column, memory=2*1024**3)
```

#### Estimating memory requirements

You can follow these simple steps to estimate the memory requirements for your operation:

In case you have a historical job that you can use as a reference, you can:

1. Look at the memory consumption by component chart to find the task's total memory consumption
2. Look at the total number of running tasks 
3. Divide the total memory consumption by the number of running tasks to get the memory budget per task

In case you don't have a historical job to use as a reference, you can:
1. Create a worst-case batch into memory, as an example:
   1. given the schema of the dataset, create a batch that is as large as possible
2. run a task with the UDF
3. measure the memory consumption of the task either using the dashboard or using a tool like `psutil`


### Pre-fetching data for Stateful Transformations

For a stateful transformation, Ray Data will submit a 4x the number of tasks to the cluster. This is done to ready all task dependencies by fetching the data in advance.

For example, take this code:

In [ ]:
class StatefulTransformer:
    def __call__(self, df: pd.DataFrame) -> pd.DataFrame:
        time.sleep(60 * 5)
        return df

ds = ray.data.from_items([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
assert ds.num_blocks() == 10 
ds_adjusted = ds.map_batches(StatefulTransformer, concurrency=2, batch_format="pandas", batch_size=None)
ds_adjusted.to_pandas()

In the above example, Ray Data will submit 4x the number of tasks to the cluster (8 task total - i.e. 6 of which will be pending) to fetch the data in advance.

## 5. All-to-all data operations

All-to-all data operations require moving (shuffling) data across the cluster.

Ray Data implements two algorithms for shuffling data:
- Range-partition (sort-based) shuffling
- Hash-based shuffling

### Hash-based Shuffling
A simple and efficient shuffling method that:
- **Partition**: Splits data using a hash function (hash(key) % N) to determine which partition each row goes to
- **Push**: Sends each partition's data to dedicated aggregator actors
- **Reduce**: Combines the data in each partition into final blocks

**Best for:** Operations needing consistent key-based grouping (joins, group-by) since rows with the same key always go to the same partition.

### Range-partition Shuffling
A sorting-based approach that:
- **Sample**: Takes random samples from each block to estimate data ranges
- **Partition**: Splits blocks into partitions based on these range boundaries
- **Reduce**: Combines partitions within the same range into final blocks

**Best for:** When data is already sorted by the key you're using, as it can leverage the existing order.


The key difference is that hash-based shuffling is more general-purpose and deterministic, while range-partition shuffling can be more efficient when data is pre-sorted but requires more computation to determine the ranges.

### GroupBy

`GroupBy` is an all-to-all data operation that groups each input record to multiple output records.

To use hash-based shuffling, set the following configuration:

In [ ]:
from ray.data.context import ShuffleStrategy

ray.data.DataContext.get_current().shuffle_strategy = ShuffleStrategy.HASH_SHUFFLE

You can then perform a groupby like this:

```python
ds = ray.data.read_parquet(DATA_PATH_FEATURES, columns=["year", "month", "tolls_amount"])
ds.groupby(key=["year", "month"]).map_groups(
    lambda df: df["tolls_amount"].sum(), 
    batch_format="pandas"
).take_all()
```

### Randomization operations

Randomization operations are all-to-all operations that require redistributing data across partitions.

Take for example the `random_shuffle` operation.

```python
ds.random_shuffle().to_pandas()
```

By default, the `random_shuffle` operation will require shuffling the data across all the partitions in the cluster.

This is a very expensive operation primarily because for a complete randomization, we need to fully materialize the dataset into memory.

Algorithmically, you can describe the operation as:
- Time complexity: **O(dataset size / parallelism)**
  - because instead of waiting for the full data to be randomized, we randomize in parallel across the N partitions.
- Space complexity: **O(dataset size)**

#### Be mindful of head node resources for large all-to-all operations

Ray Data uses the driver node to keep track of the blocks in the dataset. In case of a large all-to-all operation, the driver node may run out of memory.

## 6. Materializing intermediate results

Only resort to materializing intermediate results if you have a good reason to do so.

- Your materialized data will comfortably fit into cluster memory
- You have multiple downstream pipelines that will consume the materialized data
- You want to avoid data transfer costs for ephemeral data


For example, materializing data prior to running distributed hyperparameter tuning is a good use case.


Implicit materialization is when performing:

- Shuffle operations whose downstream operation requires the full data - e.g. a `join` operation between two datasets
- A full dataset partition or re-partition (in both cases this is the definition of a materialization)



## 7. Writing Data

Similiar to reads, look for a built-in Ray IO connector for the data source See the [docs page here](https://docs.ray.io/en/latest/data/api/input_output.html#input-output)

In case you don't find a built-in connector, there are a few escape hatches you can use:
- Use `map_batches` to write data (this works if a mapping of batch:file works for your use case)
- Implement your own custom write sink. See the [docs page here](https://docs.ray.io/en/latest/data/custom-datasource-example.html)
